In [ ]:
pip install shap

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
import pickle
import shap
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix, RocCurveDisplay, ConfusionMatrixDisplay,precision_score, recall_score, f1_score, roc_curve
import matplotlib.pyplot as plt
import itertools, heapq
import pandas as pd
from itertools import count
from scipy import stats
from scipy.stats import pearsonr
from xgboost import XGBClassifier
from collections import defaultdict


df = pd.read_csv("rrt.csv")

In [ ]:
def clean_numeric_columns(df):
    df_clean = df.copy()

    for col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str)

        df_clean[col] = (
            df_clean[col]
            .str.replace(",", ".", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace(r"[^0-9.\-]", "", regex=True)
        )

        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    return df_clean

df = clean_numeric_columns(df)

print(df.dtypes)
print(df.head())


In [ ]:
df_cleaned = df.dropna()
df=df_cleaned
nan_summary = pd.DataFrame({
    "Column": df.columns,
    "NaN_Count": df.isna().sum().values
})

nan_summary

In [ ]:
for col in df.columns:
    if col != "MORTALİTE":
        corr = df[col].corr(df["MORTALİTE"])
        print(f"Correlation of {col} with MORTALİTE: {corr:.2f}")

In [ ]:
for col in df.columns:
    if col != "RRT":
        corr = df[col].corr(df["RRT"])
        print(f"Correlation of {col} with RRT: {corr:.2f}")

In [ ]:
for col in df.columns:
    if col != "MORTALİTE" and pd.api.types.is_numeric_dtype(df[col]):
        r, p_value = pearsonr(df[col], df["MORTALİTE"])
        significance = "Significant" if p_value < 0.05 else "Not significant"
        print(f"{col}: Correlation = {r:.2f}, p-value = {p_value:.4f} → {significance}")


In [ ]:
for col in df.columns:
    if col != "RRT" and pd.api.types.is_numeric_dtype(df[col]):
        r, p_value = pearsonr(df[col], df["RRT"])
        significance = "Significant" if p_value < 0.05 else "Not significant"
        print(f"{col}: Correlation = {r:.2f}, p-value = {p_value:.4f} → {significance}")

In [ ]:
def delong_roc_test(y_true, probA, probB):

    y_true = np.concatenate(y_true)
    probA = np.concatenate(probA)
    probB = np.concatenate(probB)

    auc_a = roc_auc_score(y_true, probA)
    auc_b = roc_auc_score(y_true, probB)

    diff = auc_a - auc_b

    m = sum(y_true == 1)
    n = sum(y_true == 0)
    var_a = (auc_a * (1 - auc_a) + (m - 1) * (0 - 0)**2 + (n - 1) * (0 - 0)**2) / (m * n) # simplified

    z = diff / np.sqrt(np.var(probA - probB) / len(y_true))
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))

    return auc_a, auc_b, p_value

In [ ]:
df = pd.read_csv('rrt.csv')

kf = KFold(n_splits=5, shuffle=True, random_state=42)

folds = []
for train_idx, test_idx in kf.split(df):
    folds.append((train_idx, test_idx))

with open('kfold_indices.pkl', 'wb') as f:
    pickle.dump(folds, f)

print("Folds saved successfully")

In [ ]:
results_list_rf = []
results_list_xgb = []
results_list_svm = []

In [ ]:
DATA_PATH = 'rrt.csv'
FOLDS_PATH = 'kfold_indices.pkl'
TARGET_COLUMN = 'RRT'

RF_SELECTED_FEATURES = ['SOFA', 'K', 'AISI']

def run_rf_5fold_cv():
    global results_list_rf
    results_list_rf = []

    try:
        df = pd.read_csv(DATA_PATH)
        with open(FOLDS_PATH, 'rb') as f:
            folds = pickle.load(f)
    except FileNotFoundError as e:
        print(f"Error: Ensure your data and fold files exist. {e}")
        return

    X = df[RF_SELECTED_FEATURES]
    y = df[TARGET_COLUMN]

    print(f"Starting 5-Fold CV for Random Forest...")
    print(f"Features: {RF_SELECTED_FEATURES}\n")

    for i, (train_idx, test_idx) in enumerate(folds):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        rf_model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )

        rf_model.fit(X_train, y_train)

        y_pred = rf_model.predict(X_test)
        y_score = rf_model.predict_proba(X_test)[:, 1]

        auc = roc_auc_score(y_test, y_score)
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        fpr, tpr, _ = roc_curve(y_test, y_score)

        shap_values_fold = get_shap_values(rf_model, X_test, RF_SELECTED_FEATURES, model_type='tree')

        results_list_rf.append({
            "fold": i + 1,
            "auc": auc,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "fpr": fpr,
            "tpr": tpr,
            "y_true": y_test.values,
            "y_pred": y_pred,
            "y_score": y_score,
            "shap_values": shap_values_fold
        })

        print(f"Fold {i+1} | AUC: {auc:.4f} | Accuracy: {acc:.4f}")

    df_metrics = pd.DataFrame(results_list_rf)

    summary_metrics = ["auc", "accuracy", "precision", "recall", "f1"]

    print("\n" + "="*40)
    print("FINAL CROSS-VALIDATION SUMMARY (5 FOLDS)")
    print("="*40)
    summary_mean = df_metrics[summary_metrics].mean()
    summary_std = df_metrics[summary_metrics].std()

    for metric in summary_metrics:
        print(f"{metric:10}: {summary_mean[metric]:.4f} +- {summary_std[metric]:.4f}")

    return df_metrics

if __name__ == "__main__":
    final_results = run_rf_5fold_cv()

In [ ]:
rf_results_with_shap_df = run_rf_5fold_cv()

In [ ]:
XG_SELECTED_FEATURES = ['SOFA', 'K', 'SII']

def run_xgboost_5fold_cv():
    global results_list_xgb
    results_list_xgb = []
    try:
        df = pd.read_csv(DATA_PATH)
        with open(FOLDS_PATH, 'rb') as f:
            folds = pickle.load(f)
    except FileNotFoundError as e:
        print(f"Error: Required files not found. {e}")
        return

    X = df[XG_SELECTED_FEATURES]
    y = df[TARGET_COLUMN]

    print(f"Starting 5-Fold CV for XGBoost...")
    print(f"Features used: {XG_SELECTED_FEATURES}\n")

    for i, (train_idx, test_idx) in enumerate(folds):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        xgb_model = XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            eval_metric="logloss"
        )

        xgb_model.fit(X_train, y_train)

        y_pred = xgb_model.predict(X_test)
        y_score = xgb_model.predict_proba(X_test)[:, 1]

        auc_val = roc_auc_score(y_test, y_score)
        acc_val = accuracy_score(y_test, y_pred)
        prec_val = precision_score(y_test, y_pred, zero_division=0)
        rec_val = recall_score(y_test, y_pred, zero_division=0)
        f1_val = f1_score(y_test, y_pred, zero_division=0)

        fpr, tpr, _ = roc_curve(y_test, y_score)

        shap_values_fold = get_shap_values(xgb_model, X_test, XG_SELECTED_FEATURES, model_type='tree')

        results_list_xgb.append({
            "fold": i + 1,
            "auc": auc_val,
            "accuracy": acc_val,
            "precision": prec_val,
            "recall": rec_val,
            "f1": f1_val,
            "fpr": fpr,
            "tpr": tpr,
            "y_true": y_test.values,
            "y_pred": y_pred,
            "y_score": y_score,
            "shap_values": shap_values_fold
        })

        print(f"Fold {i+1} | AUC: {auc_val:.4f} | Accuracy: {acc_val:.4f}")

    df_metrics = pd.DataFrame(results_list_xgb)

    summary_metrics = ["auc", "accuracy", "precision", "recall", "f1"]

    print("\n" + "="*40)
    print("XGBOOST FINAL CROSS-VALIDATION SUMMARY")
    print("="*40)
    summary_mean = df_metrics[summary_metrics].mean()
    summary_std = df_metrics[summary_metrics].std()
    for metric in summary_metrics:
        print(f"{metric:10}: {summary_mean[metric]:.4f} +- {summary_std[metric]:.4f}")

    return df_metrics

if __name__ == "__main__":
    xgb_results = run_xgboost_5fold_cv()

In [ ]:
xgb_results_with_shap_df = run_xgboost_5fold_cv()

In [ ]:
SVM_SELECTED_FEATURES = ['SOFA', 'K', 'SII']

def run_svm_5fold_cv():
    global results_list_svm
    results_list_svm = []

    try:
        df = pd.read_csv(DATA_PATH)
        with open(FOLDS_PATH, 'rb') as f:
            folds = pickle.load(f)
    except FileNotFoundError as e:
        print(f"Error: Required files not found. {e}")
        return

    X = df[SVM_SELECTED_FEATURES]
    y = df[TARGET_COLUMN]

    print(f"Starting 5-Fold CV for SVM (RBF Kernel)...")
    print(f"Features used: {SVM_SELECTED_FEATURES}\n")

    for i, (train_idx, test_idx) in enumerate(folds):
        X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw)
        X_test = scaler.transform(X_test_raw)

        svm_model = SVC(
            kernel="rbf",
            C=1.0,
            probability=True,
            random_state=42
        )

        svm_model.fit(X_train, y_train)

        y_pred = svm_model.predict(X_test)
        y_score = svm_model.predict_proba(X_test)[:, 1]


        auc_val = roc_auc_score(y_test, y_score)
        acc_val = accuracy_score(y_test, y_pred)
        prec_val = precision_score(y_test, y_pred, zero_division=0)
        rec_val = recall_score(y_test, y_pred, zero_division=0)
        f1_val = f1_score(y_test, y_pred, zero_division=0)

        fpr_val, tpr_val, _ = roc_curve(y_test, y_score)

        shap_values_fold = get_shap_values(svm_model, X_test, SVM_SELECTED_FEATURES, model_type='kernel')

        results_list_svm.append({
            "fold": i + 1,
            "auc": auc_val,
            "accuracy": acc_val,
            "precision": prec_val,
            "recall": rec_val,
            "f1": f1_val,
            "fpr": fpr_val,
            "tpr": tpr_val,
            "y_true": y_test.values,
            "y_pred": y_pred,
            "y_score": y_score,
            "shap_values": shap_values_fold
        })

        print(f"Fold {i+1} | AUC: {auc_val:.4f} | Accuracy: {acc_val:.4f}")

    df_metrics = pd.DataFrame(results_list_svm)
    summary_metrics = ["auc", "accuracy", "precision", "recall", "f1"]

    print("\n" + "="*40)
    print("SVM FINAL CROSS-VALIDATION SUMMARY")
    print("="*40)
    summary_mean = df_metrics[summary_metrics].mean()
    summary_std = df_metrics[summary_metrics].std()
    for metric in summary_metrics:
        print(f"{metric:10}: {summary_mean[metric]:.4f} +- {summary_std[metric]:.4f}")

    return df_metrics

if __name__ == "__main__":
    svm_results_df = run_svm_5fold_cv()

In [ ]:
svm_results_with_shap_df = run_svm_5fold_cv()

In [ ]:
svm_results_df = run_svm_5fold_cv()

In [ ]:
def aggregate_shap_values(results_df):
    all_shap_values = []

    for index, row in results_df.iterrows():
        all_shap_values.append(row['shap_values'])

    aggregated_shap_values = np.vstack(all_shap_values)

    return aggregated_shap_values

print("SHAP aggregation utility function defined.")

In [ ]:
print("\n" + "="*40)
print("SHAP Summary Plot for Random Forest")
print("="*40)
aggregated_rf_shap_values = aggregate_shap_values(rf_results_with_shap_df)

df = pd.read_csv(DATA_PATH)
X_sample_rf = df[RF_SELECTED_FEATURES].iloc[rf_results_with_shap_df.iloc[0]['y_true'].index.values]

if isinstance(aggregated_rf_shap_values, list) and len(aggregated_rf_shap_values) > 1:
    shap.summary_plot(aggregated_rf_shap_values[1], X_sample_rf, feature_names=RF_SELECTED_FEATURES, show=False)
else:
    shap.summary_plot(aggregated_rf_shap_values, X_sample_rf, feature_names=RF_SELECTED_FEATURES, show=False)

plt.title('Random Forest SHAP Summary Plot')
plt.show()

print("\n" + "="*40)
print("SHAP Summary Plot for XGBoost")
print("="*40)
aggregated_xgb_shap_values = aggregate_shap_values(xgb_results_with_shap_df)

X_sample_xgb = df[O_SELECTED_FEATURES].iloc[xgb_results_with_shap_df.iloc[0]['y_true'].index.values]

if isinstance(aggregated_xgb_shap_values, list) and len(aggregated_xgb_shap_values) > 1:
    shap.summary_plot(aggregated_xgb_shap_values[1], X_sample_xgb, feature_names=O_SELECTED_FEATURES, show=False)
else:
    shap.summary_plot(aggregated_xgb_shap_values, X_sample_xgb, feature_names=O_SELECTED_FEATURES, show=False)

plt.title('XGBoost SHAP Summary Plot')
plt.show()

print("\n" + "="*40)
print("SHAP Summary Plot for SVM")
print("="*40)
aggregated_svm_shap_values = aggregate_shap_values(svm_results_with_shap_df)

X_sample_svm = df[O_SELECTED_FEATURES].iloc[svm_results_with_shap_df.iloc[0]['y_true'].index.values]

if isinstance(aggregated_svm_shap_values, list) and len(aggregated_svm_shap_values) > 1:
    shap.summary_plot(aggregated_svm_shap_values[1], X_sample_svm, feature_names=O_SELECTED_FEATURES, show=False)
else:
    shap.summary_plot(aggregated_svm_shap_values, X_sample_svm, feature_names=O_SELECTED_FEATURES, show=False)

plt.title('SVM SHAP Summary Plot')
plt.show()

In [ ]:
svm_results_with_shap_df = run_svm_5fold_cv()

In [ ]:
xgb_results_df = run_xgboost_5fold_cv()

In [ ]:
rf_results_df = run_rf_5fold_cv()

In [ ]:
rf_results_with_shap_df = run_rf_5fold_cv()

In [ ]:
y_true_svm = svm_results_df['y_true'].tolist()
y_score_svm = svm_results_df['y_score'].tolist()

y_score_xgb = xgb_results_df['y_score'].tolist()

auc_svm, auc_xgb, p_val = delong_roc_test(y_true_svm, y_score_svm, y_score_xgb)

print("="*40)
print("DELONG TEST: SVM vs XGBOOST")
print("="*40)
print(f"SVM AUC:     {auc_svm:.4f}")
print(f"XGBoost AUC: {auc_xgb:.4f}")
print(f"P-Value:     {p_val:.4f}")

if p_val < 0.05:
    winner = "SVM" if auc_svm > auc_xgb else "XGBoost"
    print(f"\nRESULT: Statistically Significant. {winner} is better.")
else:
    print("\nRESULT: No Significant Difference. Both models perform similarly.")

In [ ]:
y_true_svm = svm_results_df['y_true'].tolist()
y_score_svm = svm_results_df['y_score'].tolist()

y_score_rf = rf_results_df['y_score'].tolist()

auc_svm, auc_rf, p_val = delong_roc_test(y_true_svm, y_score_svm, y_score_rf)

print("="*40)
print("DELONG TEST: SVM vs RF")
print("="*40)
print(f"SVM AUC:     {auc_svm:.4f}")
print(f"RF AUC: {auc_rf:.4f}")
print(f"P-Value:     {p_val:.4f}")

if p_val < 0.05:
    winner = "SVM" if auc_svm > auc_rf else "RF"
    print(f"\nRESULT: Statistically Significant. {winner} is better.")
else:
    print("\nRESULT: No Significant Difference. Both models perform similarly.")

In [ ]:
y_true_xgb = xgb_results_df['y_true'].tolist()
y_score_xgb = xgb_results_df['y_score'].tolist()

y_score_rf = rf_results_df['y_score'].tolist()

auc_xgb, auc_rf, p_val = delong_roc_test(y_true_xgb, y_score_xgb, y_score_rf)

print("="*40)
print("DELONG TEST: XGBoost vs RF")
print("="*40)
print(f"SVM AUC:     {auc_xgb:.4f}")
print(f"RF AUC: {auc_rf:.4f}")
print(f"P-Value:     {p_val:.4f}")

if p_val < 0.05:
    winner = "XGBoost" if auc_svm > auc_rf else "RF"
    print(f"\nRESULT: Statistically Significant. {winner} is better.")
else:
    print("\nRESULT: No Significant Difference. Both models perform similarly.")

In [ ]:
all_y_true = np.concatenate(svm_results_df['y_true'].values)
all_svm_scores = np.concatenate(svm_results_df['y_score'].values)

fpr, tpr, thresholds = roc_curve(all_y_true, all_svm_scores)

best_thresh, best_j = find_optimal_threshold(fpr, tpr, thresholds)

print(f"Optimal Threshold for SVM: {best_thresh:.4f}")
print(f"Max Youden Index (J): {best_j:.4f}")

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

def get_feature_thresholds(df, target_col, feature_list):
    feature_results = []

    for feat in feature_list:
        fpr, tpr, thresholds = roc_curve(df[target_col], df[feat])

        j_scores = tpr - fpr
        best_idx = np.argmax(j_scores)

        best_thresh = thresholds[best_idx]
        best_j = j_scores[best_idx]

        feature_results.append({
            "Feature": feat,
            "Optimal_Threshold": round(best_thresh, 4),
            "Max_Youden_J": round(best_j, 4)
        })

    return pd.DataFrame(feature_results)

In [ ]:
my_features = ['SOFA', 'K', 'APACHE II','CHARLSON', 'SII', 'SIRI', 'AISI'] # or ['A', 'B', 'C']

feature_thresholds_df = get_feature_thresholds(df, 'RRT', my_features)

print("OPTIMAL THRESHOLDS PER FEATURE")
print(feature_thresholds_df)